第 1 步：先做一个最小 PyTorch 模型

In [ ]:
import torch
import torch.nn as nn

class MiniModel(nn.Module):
    def forward(self, x):
        y = torch.relu(x)
        z = y + 1.0
        return z

model = MiniModel().eval()
x = torch.tensor([[-1.0, 0.5, 2.0]], dtype=torch.float32)

torch.onnx.export(
    model,
    x,
    "work/mini.onnx",
    input_names=["input"],
    output_names=["output"],
    opset_version=13
)

print("exported: work/mini.onnx")

把 ONNX 里的 Add 改成 custom op

In [2]:
import onnx
from onnx import helper

model = onnx.load("work/mini.onnx")
graph = model.graph

new_nodes = []
for node in graph.node:
    if node.op_type == "Add":
        custom_node = helper.make_node(
            "MyScale",
            inputs=[node.input[0]],
            outputs=list(node.output),
            domain="my.custom",
            alpha=2.0
        )
        new_nodes.append(custom_node)
    else:
        new_nodes.append(node)

del graph.node[:]
graph.node.extend(new_nodes)

# 清理未被引用的 Constant 节点
used_inputs = set()
for node in graph.node:
    used_inputs.update(node.input)

filtered_nodes = []
for node in graph.node:
    if node.op_type == "Constant" and all(o not in used_inputs for o in node.output):
        continue
    filtered_nodes.append(node)

del graph.node[:]
graph.node.extend(filtered_nodes)

model.opset_import.extend([helper.make_opsetid("my.custom", 1)])
onnx.save(model, "work/mini_custom.onnx")
print("saved: work/mini_custom.onnx")

saved: work/mini_custom.onnx


我们规定：

MyScale(x, alpha) = x * alpha

所以整个模型变成：

output = MyScale(Relu(x), alpha=2.0)

如果输入：

[-1.0, 0.5, 2.0]

那么：

Relu -> [0.0, 0.5, 2.0]
MyScale -> [0.0, 1.0, 4.0]

开始op-package写 OpDef XML
写 OpDef XML

你贴的文档已经说明了，OpDef 里至少要有：

Name
Input
Output
Parameter（可选）
SupportedBackend
UseDefaultTranslation

并且 custom op 至少要有一个输入和一个输出

这里的关键点：

Reference Source="ONNX"：给 converter 做 source-side 对应
UseDefaultTranslation=false：表示这是 generic custom op，不是覆盖 QNN 原生 op
SupportedBackend=CPU：先走最小闭环，不碰 HTP

In [ ]:
<OpDefCollection PackageName="MyScaleOpPackage">

  <OpDef>
    <Name>MyScale</Name>

    <Description>
      <Content>Multiply input tensor by scalar alpha</Content>
    </Description>

    <Reference Source="ONNX" Url="custom://my.custom/MyScale"></Reference>

    <Input>
      <Name>input</Name>
      <Mandatory>true</Mandatory>
      <Datatype>QNN_DATATYPE_FLOAT_32</Datatype>
      <Shape>
        <Rank>ND</Rank>
        <Layout>UNDEFINED</Layout>
      </Shape>
    </Input>

    <Output>
      <Name>output</Name>
      <Mandatory>true</Mandatory>
      <Datatype>QNN_DATATYPE_FLOAT_32</Datatype>
      <Shape>
        <Rank>ND</Rank>
        <Layout>UNDEFINED</Layout>
      </Shape>
    </Output>

    <Parameter>
      <Name>alpha</Name>
      <Mandatory>true</Mandatory>
      <Datatype>QNN_DATATYPE_FLOAT_32</Datatype>
      <Shape>
        <Rank>SCALAR</Rank>
        <Layout>UNDEFINED</Layout>
      </Shape>
      <Default>1.0</Default>
    </Parameter>

    <UseDefaultTranslation>false</UseDefaultTranslation>
    <SupportedBackend>CPU</SupportedBackend>
  </OpDef>

</OpDefCollection>

生成 custom op package 骨架

QNN/QAIRT 文档里有 qnn-op-package-generator 的说明，用来根据 op package 配置生成包骨架。

In [11]:
import subprocess

subprocess.run([
    "qnn-op-package-generator",
    "-p", "custom_op/custom_Op.xml",
    "--output_path", "work/op_package"
], check=True)

ERROR: OpDef XML does not match Schema
Element 'Name': This element is not expected. Expected is ( OpDef )., line 4


Traceback (most recent call last):
  File "/opt/qairt/2.44.0.260225/bin/x86_64-linux-clang/qnn-op-package-generator", line 24, in <module>
    qnn_code_generator.setup()
  File "/opt/qairt/2.44.0.260225/lib/python/qti/aisw/op_package_generator/parser.py", line 51, in setup
    self.__generator.parse_config(self.config_path, self.output_path, self.converter_op_package)
  File "/opt/qairt/2.44.0.260225/lib/python/qti/aisw/op_package_generator/generator.py", line 141, in parse_config
    config_path, op_collection = self.parse_config_to_op_def_collection(config_path)
  File "/opt/qairt/2.44.0.260225/lib/python/qti/aisw/op_package_generator/generator.py", line 114, in parse_config_to_op_def_collection
    xml_instance = self.__translator(config_path, self.SCHEMA)
  File "/opt/qairt/2.44.0.260225/lib/python/qti/aisw/op_package_generator/translator/op_def_translator.py", line 136, in __init__
    self.__validate_xml_against_schema()
  File "/opt/qairt/2.44.0.260225/lib/python/qti/aisw/op_pac

CalledProcessError: Command '['qnn-op-package-generator', '-p', 'custom_op/custom_Op.xml', '--output_path', 'work/op_package']' returned non-zero exit status 1.